In [ ]:
#Install AgentNova, Ollama, Cloudflared
!sudo apt-get install zstd pciutils nano
!pip install git+https://github.com/VTSTech/AgentNova.git
# Install Ollama
!sudo curl -fsSL https://ollama.com/install.sh | sh
# Install cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared
!chmod +x /tmp/cloudflared

In [2]:
# 2. Start Ollama in the background
import subprocess, os, time
# We use OLLAMA_HOST=0.0.0.0 so the tunnel can find it
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_API_KEY'] = 'ollama-local'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '262144'
subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=open('ollama.log', 'w'))
time.sleep(1)

In [ ]:
#Cloudflare Tunnel Ollama
import subprocess
import re
import time

# Start cloudflared
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait and capture the tunnel URL
tunnel_url = None
for _ in range(30):  # 30 second timeout
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print(f"✅ Tunnel URL: {tunnel_url}")
else:
    print("❌ Failed to get tunnel URL")

In [ ]:

#Pull Ollama Models
#!ollama pull nemotron-3-nano:4b
#!ollama pull AgentricAi/AgentricAI_TLM:latest
#!ollama pull DedeProgames/orion:2b
#!ollama pull driaforall/tiny-agent-a:1.5b
#!ollama pull deepseek-coder:1.3b
!ollama pull tinyllama:1.1b
!ollama pull tinydolphin:1.1b
!ollama pull granite3.1-moe:1b
!ollama pull llama3.2:1b
!ollama pull nchapman/dolphin3.0-llama3:1b
!ollama pull qwen2.5-coder:0.5b-instruct-q4_k_m
!ollama pull nchapman/dolphin3.0-qwen2.5:0.5b
!ollama pull qwen:0.5b
!ollama pull qwen3:0.6b
!ollama pull qwen2.5:0.5b
!ollama pull granite4:350m
!ollama pull functiongemma:270m
!ollama pull gemma3:270m
!ollama list

In [ ]:
#Pull BitNet Model
# 1. Mount your Drive so we can save the success
from google.colab import drive
drive.mount('/content/drive')

# 2. Get the official model (skipping the broken conversion scripts)
!mkdir -p /content/BitNet/models/BitNet-b1.58-2B-4T
!wget -O /content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf \
    "https://huggingface.co/microsoft/bitnet-b1.58-2B-4T-gguf/resolve/main/ggml-model-i2_s.gguf"
!ls -lh /content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf
# 3. Save it to your backup so you never have to do this again
!cp /content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf /content/drive/MyDrive/BitNet_Backup/

In [ ]:
#BitNet llama in background
import subprocess, threading, time, os
!pkill llama-server
# 1. Prepare the Binary
!chmod +x /content/BitNet/build/bin/llama-server

# 2. Define the Launch Command
# We use the full path to ensure no "file not found" errors
#model_path = "/content/BitNet/models/Falcon3-1B-Instruct-1.58bit/ggml-model-i2_s.gguf"
model_path = "/content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf"
bin_path = "/content/BitNet/build/bin/llama-server"

cmd = [
    bin_path,
    "-m", model_path,
    "--host", "0.0.0.0",
    "--port", "8765",
    "--ctx-size", "2048",
    "--no-mmap"
]

# 3. Launch llama-server in the background
# We redirect output to a log file so it doesn't spam the cell
with open('llama_server.log', 'w') as log_file:
    server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("⏳ llama-server is warming up in the background...")
time.sleep(1) # Give the model a moment to load into RAM

In [ ]:
#Cloudflare Tunnel Ollama (BitNet)
import subprocess
import re
import time

# Start cloudflared
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8765'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait and capture the tunnel URL
tunnel_url = None
for _ in range(30):  # 30 second timeout
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print(f"✅ Tunnel URL: {tunnel_url}")
else:
    print("❌ Failed to get tunnel URL")

In [ ]:
# --- COMPILE BITNET (THE FINAL VERSION) ---
import os, shutil

# 1. Clean & Setup
%cd /content
!sudo apt-get update -y && sudo apt-get install -y clang cmake
!rm -rf /content/BitNet
!git clone --recursive https://github.com/microsoft/BitNet.git
%cd /content/BitNet

# 2. SURGERY: Fix the code bug
!sed -i '811s/int8_t \* y_col/const int8_t \* y_col/' /content/BitNet/src/ggml-bitnet-mad.cpp

# 3. CODEGEN: Create the missing header file
# We'll use the 2B model parameters since that's what you're running
print("Running BitNet CodeGen...")
!python3 utils/codegen_tl2.py --model bitnet_b1_58-3B --BM 160,320,320 --BK 96,96,96 --bm 32,32,32
# Note: codegen creates the header in the root or include/
!mkdir -p include
!cp bitnet-lut-kernels.h include/ 2>/dev/null || cp utils/bitnet-lut-kernels.h include/ 2>/dev/null || echo "Header already in place"

# 4. STATIC BUILD CONFIG
os.environ["CC"] = "clang"
os.environ["CXX"] = "clang++"

!mkdir -p build
%cd build
!cmake .. \
    -DBUILD_SHARED_LIBS=OFF \
    -DLLAMA_BUILD_SERVER=ON \
    -DCMAKE_C_FLAGS="-I/content/BitNet/include" \
    -DCMAKE_CXX_FLAGS="-I/content/BitNet/include"

# 5. EXECUTE BUILD
!make llama-server -j$(nproc)

# 6. VERIFY & BACKUP
if os.path.exists("/content/BitNet/build/bin/llama-server"):
    print("\n✅ SUCCESS! Binary is standalone.")
    !ldd /content/BitNet/build/bin/llama-server
    !cp /content/BitNet/build/bin/llama-server /content/drive/MyDrive/BitNet_Backup/llama-server-static
    print("🚀 Standalone binary saved to Google Drive.")
else:
    print("\n❌ Build failed. Checking if header exists...")
    !ls -l /content/BitNet/include/bitnet-lut-kernels.h

In [ ]:
#backup binaries
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for your BitNet assets
!mkdir -p /content/drive/MyDrive/BitNet_Backup

# 1. Save the successfully compiled binaries
!tar -cvzf /content/drive/MyDrive/BitNet_Backup/bitnet_binaries.tar.gz -C /content/BitNet/build/bin .

# 2. Save the model weights (so you don't have to download 0.4GB again)
!cp -r /content/BitNet/models /content/drive/MyDrive/BitNet_Backup/

In [ ]:
#restore binaries
from google.colab import drive
import os

# 1. Mount and Setup Folders
drive.mount('/content/drive')
!mkdir -p /content/BitNet/build/bin
!mkdir -p /content/BitNet/models

# 2. Extract Binaries and Models
!tar -xvzf /content/drive/MyDrive/BitNet_Backup/bitnet_binaries.tar.gz -C /content/BitNet/build/bin
!cp -r /content/drive/MyDrive/BitNet_Backup/models/* /content/BitNet/models/

In [ ]:
!curl http://127.0.0.1:11434
!ollama list